In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf

In [12]:
data = pd.read_csv('../bitcoin_data/BTCUSDT_1d_2020_present.csv')
data.head()

,date,open,high,low,close,volume,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,taker_buy_quote_asset_volume
0,2020-01-01,7195.24,7255.0,7175.15,7200.85,16792.388165,1.212145e+08,194010.0,8946.955535,6.459779e+07
1,2020-01-02,7200.77,7212.5,6924.74,6965.71,31951.483932,2.259823e+08,302667.0,15141.611340,1.070608e+08
2,2020-01-03,6965.49,7405.0,6871.04,7344.96,68428.500451,4.950986e+08,519854.0,35595.496273,2.577131e+08
3,2020-01-04,7345.00,7404.0,7272.21,7354.11,29987.974977,2.198742e+08,279370.0,16369.382248,1.200351e+08
4,2020-01-05,7354.19,7495.0,7318.00,7358.75,38331.085604,2.848487e+08,329209.0,19455.369564,1.446001e+08


In [13]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2361 entries, 0 to 2360
Data columns (total 10 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   date                          2361 non-null   str    
 1   open                          2361 non-null   float64
 2   high                          2361 non-null   float64
 3   low                           2361 non-null   float64
 4   close                         2361 non-null   float64
 5   volume                        2361 non-null   float64
 6   quote_asset_volume            2361 non-null   float64
 7   number_of_trades              2361 non-null   float64
 8   taker_buy_base_asset_volume   2361 non-null   float64
 9   taker_buy_quote_asset_volume  2361 non-null   float64
dtypes: float64(9), str(1)
memory usage: 184.6 KB


In [14]:
df = data.copy()

In [15]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Target: next-day percentage movement
df["target_pct"] = ((df["close"].shift(-1) - df["close"]) / df["close"]) * 100

##### Feature Engineering

In [16]:
# Candlestick structure
df["body_pct"] = (df["close"] - df["open"]) / df["open"]
df["range_pct"] = (df["high"] - df["low"]) / df["open"]

df["upper_wick_pct"] = (
    df["high"] - df[["open", "close"]].max(axis=1)
) / df["open"]

df["lower_wick_pct"] = (
    df[["open", "close"]].min(axis=1) - df["low"]
) / df["open"]

df["close_position"] = (df["close"] - df["low"]) / (df["high"] - df["low"])

# Returns
df["return_1d"] = df["close"].pct_change()
df["return_3d"] = df["close"].pct_change(3)
df["return_7d"] = df["close"].pct_change(7)
df["return_14d"] = df["close"].pct_change(14)

# Volume
df["volume_change"] = df["volume"].pct_change()
df["quote_volume_change"] = df["quote_asset_volume"].pct_change()
df["log_volume"] = np.log1p(df["volume"])
df["log_quote_volume"] = np.log1p(df["quote_asset_volume"])

# Buy pressure
df["buy_base_ratio"] = df["taker_buy_base_asset_volume"] / df["volume"]
df["buy_quote_ratio"] = df["taker_buy_quote_asset_volume"] / df["quote_asset_volume"]

# Trade activity
df["avg_trade_size"] = df["volume"] / df["number_of_trades"]
df["trade_count_change"] = df["number_of_trades"].pct_change()

# Moving-average ratios
df["ma_7"] = df["close"].rolling(7).mean()
df["ma_14"] = df["close"].rolling(14).mean()
df["ma_30"] = df["close"].rolling(30).mean()

df["close_ma7_ratio"] = df["close"] / df["ma_7"]
df["close_ma14_ratio"] = df["close"] / df["ma_14"]
df["close_ma30_ratio"] = df["close"] / df["ma_30"]

# Volatility
df["volatility_7d"] = df["return_1d"].rolling(7).std()
df["volatility_14d"] = df["return_1d"].rolling(14).std()
df["volatility_30d"] = df["return_1d"].rolling(30).std()

* Lag Features

In [17]:
def create_lag_features(data, variables, max_lag=30):
    lagged_df = data.copy()
    
    for var in variables:
        for lag in range(1, max_lag + 1):
            lagged_df[f"lag_{var}_{lag}"] = data[var].shift(lag)
    
    return lagged_df

In [18]:
base_features = [
    "body_pct",
    "range_pct",
    "upper_wick_pct",
    "lower_wick_pct",
    "close_position",
    "return_1d",
    "return_3d",
    "return_7d",
    "return_14d",
    "volume_change",
    "quote_volume_change",
    "log_volume",
    "log_quote_volume",
    "buy_base_ratio",
    "buy_quote_ratio",
    "avg_trade_size",
    "trade_count_change",
    "close_ma7_ratio",
    "close_ma14_ratio",
    "close_ma30_ratio",
    "volatility_7d",
    "volatility_14d",
    "volatility_30d"
]

df_lagged = create_lag_features(df, base_features, max_lag=30)

/var/folders/g2/z2xpfp7x5hvgkf6rx1kgrx4w0000gn/T/ipykernel_19536/2121596265.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  lagged_df[f"lag_{var}_{lag}"] = data[var].shift(lag)


In [19]:
df_lagged = df_lagged.replace([np.inf, -np.inf], np.nan)
df_lagged = df_lagged.dropna().reset_index(drop=True)

train_df = df_lagged[df_lagged["date"] < "2024-01-01"]
val_df = df_lagged[
    (df_lagged["date"] >= "2024-01-01") & 
    (df_lagged["date"] < "2025-01-01")
]
test_df = df_lagged[df_lagged["date"] >= "2025-01-01"]

target = "target_pct"

In [20]:
lag_feature_cols = [col for col in df_lagged.columns if col.startswith("lag_")]

correlation_results = []

for col in lag_feature_cols:
    pearson_corr = train_df[col].corr(train_df[target], method="pearson")
    spearman_corr = train_df[col].corr(train_df[target], method="spearman")
    
    # Example column name: lag_return_1d_7
    temp = col.replace("lag_", "")
    original_feature = "_".join(temp.split("_")[:-1])
    lag_number = int(temp.split("_")[-1])
    
    correlation_results.append({
        "original_feature": original_feature,
        "lag_feature": col,
        "lag": lag_number,
        "pearson_corr": pearson_corr,
        "spearman_corr": spearman_corr,
        "abs_pearson": abs(pearson_corr),
        "abs_spearman": abs(spearman_corr)
    })

lag_corr_df = pd.DataFrame(correlation_results)

lag_corr_df = lag_corr_df.sort_values("abs_spearman", ascending=False)

lag_corr_df.head(30)

,original_feature,lag_feature,lag,pearson_corr,spearman_corr,abs_pearson,abs_spearman
532,close_ma7_ratio,lag_close_ma7_ratio_23,23,0.063859,0.073201,0.063859,0.073201
396,buy_base_ratio,lag_buy_base_ratio_7,7,-0.067961,-0.072053,0.067961,0.072053
426,buy_quote_ratio,lag_buy_quote_ratio_7,7,-0.067817,-0.071929,0.067817,0.071929
131,close_position,lag_close_position_12,12,0.068289,0.070832,0.068289,0.070832
389,log_quote_volume,lag_log_quote_volume_30,30,-0.048530,-0.070815,0.048530,0.070815
562,close_ma14_ratio,lag_close_ma14_ratio_23,23,0.064138,0.068498,0.064138,0.068498
202,return_3d,lag_return_3d_23,23,0.056257,0.068165,0.056257,0.068165
395,buy_base_ratio,lag_buy_base_ratio_6,6,-0.066400,-0.067835,0.066400,0.067835
425,buy_quote_ratio,lag_buy_quote_ratio_6,6,-0.066109,-0.067647,0.066109,0.067647
133,close_position,lag_close_position_14,14,0.069215,0.066607,0.069215,0.066607


In [21]:
best_lag_per_feature = (
    lag_corr_df
    .sort_values("abs_spearman", ascending=False)
    .groupby("original_feature")
    .head(1)
    .sort_values("abs_spearman", ascending=False)
)

best_lag_per_feature

,original_feature,lag_feature,lag,pearson_corr,spearman_corr,abs_pearson,abs_spearman
532,close_ma7_ratio,lag_close_ma7_ratio_23,23,0.063859,0.073201,0.063859,0.073201
396,buy_base_ratio,lag_buy_base_ratio_7,7,-0.067961,-0.072053,0.067961,0.072053
426,buy_quote_ratio,lag_buy_quote_ratio_7,7,-0.067817,-0.071929,0.067817,0.071929
131,close_position,lag_close_position_12,12,0.068289,0.070832,0.068289,0.070832
389,log_quote_volume,lag_log_quote_volume_30,30,-0.048530,-0.070815,0.048530,0.070815
562,close_ma14_ratio,lag_close_ma14_ratio_23,23,0.064138,0.068498,0.064138,0.068498
202,return_3d,lag_return_3d_23,23,0.056257,0.068165,0.056257,0.068165
453,avg_trade_size,lag_avg_trade_size_4,4,0.046263,0.059479,0.046263,0.059479
509,trade_count_change,lag_trade_count_change_30,30,-0.035326,-0.059262,0.035326,0.059262
329,quote_volume_change,lag_quote_volume_change_30,30,-0.029666,-0.055192,0.029666,0.055192


In [22]:
correlation_threshold = 0.03

selected_lag_df = best_lag_per_feature[
    best_lag_per_feature["abs_spearman"] >= correlation_threshold
]

selected_lag_features = selected_lag_df["lag_feature"].tolist()

selected_lag_features

['lag_close_ma7_ratio_23',
 'lag_buy_base_ratio_7',
 'lag_buy_quote_ratio_7',
 'lag_close_position_12',
 'lag_log_quote_volume_30',
 'lag_close_ma14_ratio_23',
 'lag_return_3d_23',
 'lag_avg_trade_size_4',
 'lag_trade_count_change_30',
 'lag_quote_volume_change_30',
 'lag_volume_change_30',
 'lag_return_7d_23',
 'lag_lower_wick_pct_7',
 'lag_close_ma30_ratio_23',
 'lag_return_1d_27',
 'lag_body_pct_27',
 'lag_return_14d_23',
 'lag_upper_wick_pct_6',
 'lag_range_pct_22',
 'lag_volatility_7d_18',
 'lag_log_volume_30']

* Rolling Mean 

In [23]:
rolling_variables = [
    "return_1d",
    "volume_change",
    "quote_volume_change",
    "buy_base_ratio",
    "buy_quote_ratio",
    "volatility_7d"
]

windows = [3, 7, 14, 30]

rolling_features = []

for var in rolling_variables:
    for window in windows:
        new_col = f"rolling_mean_{var}_{window}"
        df_lagged[new_col] = df_lagged[var].rolling(window).mean()
        rolling_features.append(new_col)

* Cyclical Time Features

In [24]:
df_lagged["day_of_week"] = df_lagged["date"].dt.dayofweek
df_lagged["month"] = df_lagged["date"].dt.month

df_lagged["day_sin"] = np.sin(2 * np.pi * df_lagged["day_of_week"] / 7)
df_lagged["day_cos"] = np.cos(2 * np.pi * df_lagged["day_of_week"] / 7)

df_lagged["month_sin"] = np.sin(2 * np.pi * df_lagged["month"] / 12)
df_lagged["month_cos"] = np.cos(2 * np.pi * df_lagged["month"] / 12)

cyclical_features = [
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos"
]

* Final Feature set

In [25]:
df_lagged = df_lagged.replace([np.inf, -np.inf], np.nan)
df_lagged = df_lagged.dropna().reset_index(drop=True)

train_df = df_lagged[df_lagged["date"] < "2024-01-01"]
val_df = df_lagged[
    (df_lagged["date"] >= "2024-01-01") & 
    (df_lagged["date"] < "2025-01-01")
]
test_df = df_lagged[df_lagged["date"] >= "2025-01-01"]

selected_features = (
    base_features +
    selected_lag_features +
    rolling_features +
    cyclical_features
)

X_train = train_df[selected_features]
y_train = train_df[target]

X_val = val_df[selected_features]
y_val = val_df[target]

X_test = test_df[selected_features]
y_test = test_df[target]

* XGBoost Model

In [26]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

val_pred = xgb_model.predict(X_val)

print("Validation MAE:", mean_absolute_error(y_val, val_pred))
print("Validation RMSE:", np.sqrt(mean_squared_error(y_val, val_pred)))
print("Validation R2:", r2_score(y_val, val_pred))

Validation MAE: 2.0670075690333434
Validation RMSE: 2.8287575926215864
Validation R2: -0.045976803772672215


In [27]:
train_val_df = pd.concat([train_df, val_df], axis=0)

X_train_val = train_val_df[selected_features]
y_train_val = train_val_df[target]

final_xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42
)

final_xgb_model.fit(X_train_val, y_train_val)

test_pred = final_xgb_model.predict(X_test)

print("Test MAE:", mean_absolute_error(y_test, test_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, test_pred)))
print("Test R2:", r2_score(y_test, test_pred))

Test MAE: 1.6776544448489252
Test RMSE: 2.35618062676766
Test R2: -0.02502077790155033
